In [1]:
# [CELL 1] STUDENT CONFIGURATION
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Gabilan"
STUDENT_ID = "TUPM-26-2636"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

# Parameters required by the appendix exercises
N_FEATURES_EX1 = 5 + (SEED_DIGIT % 3)      # Exercise 1, requirement 2
N_PER_GROUP_TARGET = 20 + ID_SUM           # Exercise 2, requirement 2
N_BOOT = 200 + 10 * SEED_DIGIT             # Exercise 3, requirement 4
Z_THRESHOLD = 3.5

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)
print("Ex1 features:", N_FEATURES_EX1, "| Ex2 target per group:", N_PER_GROUP_TARGET, "| Ex3 bootstrap reps:", N_BOOT)

Student: Gabilan TUPM-26-2636
Seed Digit: 6
ID Sum: 25
Random Seed: 262636
Verification Value: 181
Ex1 features: 5 | Ex2 target per group: 45 | Ex3 bootstrap reps: 260


In [2]:
# [CELL 2] WORKSPACE AND DATA LOADING
WORK_DIR = Path.cwd()

# data/ may sit inside this folder or one level up (the main activity repo)
DATA_DIR = None
for candidate in (WORK_DIR / "data", WORK_DIR.parent / "data"):
    if candidate.is_dir():
        DATA_DIR = candidate
        break
assert DATA_DIR is not None, "Could not find the data folder."

OUTPUT_DIR = WORK_DIR   # outputs are saved in this exercises folder

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
for path in (measurement_path, label_path):
    assert path.is_file(), f"Missing file: {path}"

raw_measurements = np.genfromtxt(measurement_path, dtype=float)
labels = np.loadtxt(label_path, usecols=0, dtype=int)

assert raw_measurements.ndim == 2
assert len(labels) == len(raw_measurements)
assert np.isin(labels, [-1, 1]).all()

n_records, n_features = raw_measurements.shape
feature_names = np.array([f"Feature_{i:03d}" for i in range(n_features)])

# Usable = not entirely missing and not constant over observed values
missing_mask = np.isnan(raw_measurements)
observed_counts = (~missing_mask).sum(axis=0)
all_missing = observed_counts == 0
obs_min = np.where(missing_mask, np.inf, raw_measurements).min(axis=0)
obs_max = np.where(missing_mask, -np.inf, raw_measurements).max(axis=0)
constant = (observed_counts > 0) & (obs_min == obs_max)
usable_mask = ~(all_missing | constant)
usable_idx = np.flatnonzero(usable_mask)

print("Shape:", raw_measurements.shape)
print("Pass:", (labels == -1).sum(), "| Fail:", (labels == 1).sum())
print("Entirely missing:", all_missing.sum(), "| Constant:", constant.sum(), "| Usable:", len(usable_idx))

Shape: (1567, 590)
Pass: 1463 | Fail: 104
Entirely missing: 0 | Constant: 116 | Usable: 474


In [3]:
# [CELL 3] EX1 - SELECTION, STATISTICS, ROBUST STANDARDIZATION
rng1 = np.random.default_rng(RANDOM_SEED)
chosen1 = np.sort(rng1.choice(usable_idx, size=N_FEATURES_EX1, replace=False))
selected1 = feature_names[chosen1]
X1 = raw_measurements[:, chosen1]

missing_pct = 100 * np.isnan(X1).mean(axis=0)
means1 = np.nanmean(X1, axis=0)
medians1 = np.nanmedian(X1, axis=0)
sds1 = np.nanstd(X1, axis=0, ddof=1)
obs1 = (~np.isnan(X1)).sum(axis=0)

abs_dev = np.abs(X1 - medians1)
mad1 = np.nanmedian(abs_dev, axis=0)
mean_ad1 = np.nanmean(abs_dev, axis=0)
scale1 = np.where(mad1 > 0, 1.4826 * mad1, 1.253314 * mean_ad1)
scale1 = np.where(scale1 > 0, scale1, np.nan)      # last-resort safety

robust_z = (X1 - medians1) / scale1
flags1 = np.abs(robust_z) > Z_THRESHOLD            # NaN compares False
flag_counts = flags1.sum(axis=0)
flag_pct = 100 * flag_counts / obs1

quality = pd.DataFrame({
    "Feature": selected1,
    "Observed_Count": obs1,
    "Missing_Pct": missing_pct,
    "Mean": means1,
    "Median": medians1,
    "Sample_SD": sds1,
    "MAD": mad1,
    "Zero_MAD_Fallback": mad1 == 0,
    "Flagged_Count": flag_counts,
    "Flagged_Pct": flag_pct,
})
quality = quality.sort_values(["Flagged_Pct", "Feature"], ascending=[False, True]).reset_index(drop=True)
quality.insert(0, "Rank", np.arange(1, len(quality) + 1))

quality_path = OUTPUT_DIR / "measurement_quality.csv"
quality.to_csv(quality_path, index=False)

print(quality.round(4).to_string(index=False))

 Rank     Feature  Observed_Count  Missing_Pct     Mean   Median  Sample_SD     MAD  Zero_MAD_Fallback  Flagged_Count  Flagged_Pct
    1 Feature_473            1560       0.4467  39.4268  34.2468    22.4571 10.8067              False             51       3.2692
    2 Feature_338            1560       0.4467   3.0376   2.5609     5.6450  0.7257              False             25       1.6026
    3 Feature_115            1567       0.0000 747.3838 750.8614    48.9492 27.4721              False             21       1.3401
    4 Feature_011            1565       0.1276   0.9644   0.9658     0.0125  0.0065              False              5       0.3195
    5 Feature_050            1566       0.0638 632.2542 631.3709     8.6440  6.1545              False              1       0.0639


In [4]:
# [CELL 4] EX1 - VERIFY VECTORIZED RESULTS (pandas + plain loop)
dfX = pd.DataFrame(X1, columns=selected1)
med_pd = dfX.median()
dev_pd = (dfX - med_pd).abs()
scale_pd = pd.Series(np.where(dev_pd.median() > 0, 1.4826 * dev_pd.median(), 1.253314 * dev_pd.mean()),
                     index=dfX.columns)
flags_pd = ((dfX - med_pd) / scale_pd).abs() > Z_THRESHOLD

loop_counts = []
for j in range(X1.shape[1]):
    count = 0
    for v in X1[:, j]:
        if not np.isnan(v) and abs(v - medians1[j]) / scale1[j] > Z_THRESHOLD:
            count += 1
    loop_counts.append(count)

ex1_verified = bool(
    np.allclose(means1, dfX.mean().to_numpy())
    and np.allclose(medians1, med_pd.to_numpy())
    and np.allclose(sds1, dfX.std(ddof=1).to_numpy())
    and np.allclose(missing_pct, 100 * dfX.isna().mean().to_numpy())
    and np.array_equal(flags1, flags_pd.to_numpy())
    and np.array_equal(flag_counts, np.array(loop_counts))
)

hi_miss = int(np.argmax(missing_pct))
print("Selected Features:", selected1.tolist())
print("Highest Missing Percentage:", f"{missing_pct[hi_miss]:.2f}%", f"({selected1[hi_miss]})")
print("Feature with Highest Flagged Percentage:", quality.loc[0, "Feature"], f"({quality.loc[0, 'Flagged_Pct']:.4f}%)")
print("Total Flagged Observations:", int(flags1.sum()))
print("Vectorized Results Verified:", ex1_verified)

Selected Features: ['Feature_011', 'Feature_050', 'Feature_115', 'Feature_338', 'Feature_473']
Highest Missing Percentage: 0.45% (Feature_338)
Feature with Highest Flagged Percentage: Feature_473 (3.2692%)
Total Flagged Observations: 103
Vectorized Results Verified: True


In [5]:
# [CELL 5] EX2 - LABELED DATAFRAME AND BALANCED SAMPLE
df = pd.DataFrame(raw_measurements.copy(), columns=feature_names)
df.index.name = "Example_ID"                      # original record identifiers
df["Result"] = labels
df["Outcome"] = np.where(labels == -1, "Pass", "Fail")

pass_ids = df.index[df["Outcome"] == "Pass"].to_numpy()
fail_ids = df.index[df["Outcome"] == "Fail"].to_numpy()
n_per_group = min(N_PER_GROUP_TARGET, len(pass_ids), len(fail_ids))

rng2 = np.random.default_rng(RANDOM_SEED)
chosen2 = np.sort(rng2.choice(usable_idx, size=5, replace=False))
features2 = feature_names[chosen2].tolist()

sample_ids = np.sort(np.concatenate([
    rng2.choice(pass_ids, size=n_per_group, replace=False),
    rng2.choice(fail_ids, size=n_per_group, replace=False),
]))

full_medians = df[features2].median()             # medians from the full dataset
balanced = df.loc[sample_ids, features2 + ["Result", "Outcome"]].copy()
observed_before = balanced.groupby("Outcome")[features2].count()
balanced[features2] = balanced[features2].fillna(full_medians)
balanced = balanced.reset_index()                 # keeps Example_ID as a column

balanced_path = OUTPUT_DIR / "balanced_sample.csv"
balanced.to_csv(balanced_path, index=False)

print("Per-group size:", n_per_group, "| Balanced sample size:", len(balanced))
print("Selected Features:", features2)
print(balanced["Outcome"].value_counts())
print("Missing after imputation:", int(balanced[features2].isna().sum().sum()))

Per-group size: 45 | Balanced sample size: 90
Selected Features: ['Feature_011', 'Feature_050', 'Feature_115', 'Feature_338', 'Feature_473']
Outcome
Fail    45
Pass    45
Name: count, dtype: int64
Missing after imputation: 0


In [6]:
# [CELL 6] EX2 - GROUPED STATISTICS AND STANDARDIZED MEAN DIFFERENCE
stats = balanced.groupby("Outcome")[features2].agg(["count", "mean", "median", "std"])

cnt = stats.xs("count", axis=1, level=1)
mean = stats.xs("mean", axis=1, level=1)
med = stats.xs("median", axis=1, level=1)
sd = stats.xs("std", axis=1, level=1)

pooled_sd = np.sqrt(((cnt.loc["Pass"] - 1) * sd.loc["Pass"] ** 2 +
                     (cnt.loc["Fail"] - 1) * sd.loc["Fail"] ** 2) /
                    (cnt.loc["Pass"] + cnt.loc["Fail"] - 2))
abs_diff = (mean.loc["Fail"] - mean.loc["Pass"]).abs()
std_diff = (abs_diff / pooled_sd.where(pooled_sd > 0)).fillna(0.0)

comparison = pd.DataFrame({
    "Feature": features2,
    "Pass_Observed_Before_Imputation": observed_before.loc["Pass"].to_numpy(),
    "Fail_Observed_Before_Imputation": observed_before.loc["Fail"].to_numpy(),
    "Pass_Mean": mean.loc["Pass"].to_numpy(),
    "Fail_Mean": mean.loc["Fail"].to_numpy(),
    "Pass_Median": med.loc["Pass"].to_numpy(),
    "Fail_Median": med.loc["Fail"].to_numpy(),
    "Pass_SD": sd.loc["Pass"].to_numpy(),
    "Fail_SD": sd.loc["Fail"].to_numpy(),
    "Abs_Mean_Diff": abs_diff.to_numpy(),
    "Pooled_SD": pooled_sd.to_numpy(),
    "Abs_Std_Mean_Diff": std_diff.to_numpy(),
})
comparison = comparison.sort_values(["Abs_Std_Mean_Diff", "Feature"], ascending=[False, True]).reset_index(drop=True)
comparison.insert(0, "Rank", np.arange(1, len(comparison) + 1))

comparison_path = OUTPUT_DIR / "feature_comparison.csv"
comparison.to_csv(comparison_path, index=False)

# NumPy cross-check of the Pandas group statistics
is_pass = (balanced["Outcome"] == "Pass").to_numpy()
Xb = balanced[features2].to_numpy()
np_match = bool(
    np.allclose(Xb[is_pass].mean(axis=0), mean.loc["Pass"].to_numpy())
    and np.allclose(Xb[~is_pass].mean(axis=0), mean.loc["Fail"].to_numpy())
    and np.allclose(Xb[is_pass].std(axis=0, ddof=1), sd.loc["Pass"].to_numpy())
    and np.allclose(Xb[~is_pass].std(axis=0, ddof=1), sd.loc["Fail"].to_numpy())
)

print(comparison.round(4).to_string(index=False))
print()
print("Balanced Sample Size:", len(balanced))
print("Selected Features:", features2)
print("Highest-Ranked Feature:", comparison.loc[0, "Feature"])
print("Absolute Standardized Mean Difference:", f"{comparison.loc[0, 'Abs_Std_Mean_Diff']:.4f}")
print("NumPy-Pandas Statistics Match:", np_match)

 Rank     Feature  Pass_Observed_Before_Imputation  Fail_Observed_Before_Imputation  Pass_Mean  Fail_Mean  Pass_Median  Fail_Median  Pass_SD  Fail_SD  Abs_Mean_Diff  Pooled_SD  Abs_Std_Mean_Diff
    1 Feature_338                               45                               45     2.9052     3.7454       2.8268       2.7807   1.1605   5.7618         0.8402     4.1560             0.2022
    2 Feature_050                               45                               45   633.7081   632.4692     631.4291     631.7427  10.7325   8.1361         1.2389     9.5232             0.1301
    3 Feature_473                               45                               45    39.2640    41.7888      34.3512      32.6478  18.6897  26.9466         2.5248    23.1886             0.1089
    4 Feature_115                               45                               45   751.8937   748.2783     756.5663     748.3221  43.8593  38.9872         3.6154    41.4948             0.0871
    5 Feature_011        

In [7]:
# [CELL 7] EX3 - FEATURE SELECTION AND OBSERVED DIFFERENCES
pass_pos = np.flatnonzero(labels == -1)
fail_pos = np.flatnonzero(labels == 1)

obs_pass = (~np.isnan(raw_measurements[pass_pos])).sum(axis=0)
obs_fail = (~np.isnan(raw_measurements[fail_pos])).sum(axis=0)
eligible = np.flatnonzero(usable_mask & (obs_pass >= 10) & (obs_fail >= 10))

rng3 = np.random.default_rng(RANDOM_SEED)
chosen3 = np.sort(rng3.choice(eligible, size=5, replace=False))
features3 = feature_names[chosen3].tolist()

X3 = raw_measurements[:, chosen3]
medians3 = np.nanmedian(X3, axis=0)                 # full-dataset medians
X3_imp = np.where(np.isnan(X3), medians3, X3)

def group_mean_diff(X, p_rows, f_rows):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(X[f_rows], axis=0) - np.nanmean(X[p_rows], axis=0)

def pooled_sd_fn(X, p_rows, f_rows):
    Xp, Xf = X[p_rows], X[f_rows]
    n_p, n_f = (~np.isnan(Xp)).sum(axis=0), (~np.isnan(Xf)).sum(axis=0)
    v_p, v_f = np.nanvar(Xp, axis=0, ddof=1), np.nanvar(Xf, axis=0, ddof=1)
    return np.sqrt(((n_p - 1) * v_p + (n_f - 1) * v_f) / (n_p + n_f - 2))

obs_diff_avail = group_mean_diff(X3, pass_pos, fail_pos)
obs_diff_imp = group_mean_diff(X3_imp, pass_pos, fail_pos)
psd_avail = pooled_sd_fn(X3, pass_pos, fail_pos)
psd_imp = pooled_sd_fn(X3_imp, pass_pos, fail_pos)

def safe_std_diff(diff, psd):
    return np.where(psd > 0, np.abs(diff) / np.where(psd > 0, psd, 1.0), 0.0)

smd_avail = safe_std_diff(obs_diff_avail, psd_avail)
smd_imp = safe_std_diff(obs_diff_imp, psd_imp)

print("Selected Features:", features3)
print("Observed Fail-Pass diff (available-case):", obs_diff_avail)
print("Observed Fail-Pass diff (median-imputed):", obs_diff_imp)

Selected Features: ['Feature_011', 'Feature_048', 'Feature_114', 'Feature_339', 'Feature_474']
Observed Fail-Pass diff (available-case): [-0.0016 -0.2773  0.0005  0.7294  1.4715]
Observed Fail-Pass diff (median-imputed): [-0.0016 -0.2773  0.0005  0.7286  1.494 ]


In [8]:
# [CELL 8] EX3 - STRATIFIED BOOTSTRAP (with reproducibility check)
def stratified_bootstrap(seed):
    rng = np.random.default_rng(seed)
    d_avail = np.empty((N_BOOT, len(features3)))
    d_imp = np.empty((N_BOOT, len(features3)))
    for b in range(N_BOOT):
        ip = rng.choice(pass_pos, size=len(pass_pos), replace=True)   # resample within Pass
        iff = rng.choice(fail_pos, size=len(fail_pos), replace=True)  # resample within Fail
        d_avail[b] = group_mean_diff(X3, ip, iff)
        d_imp[b] = group_mean_diff(X3_imp, ip, iff)
    return d_avail, d_imp

boot_avail, boot_imp = stratified_bootstrap(RANDOM_SEED)
boot_avail_2, boot_imp_2 = stratified_bootstrap(RANDOM_SEED)
reproducible = bool(np.array_equal(boot_avail, boot_avail_2, equal_nan=True)
                    and np.array_equal(boot_imp, boot_imp_2, equal_nan=True))

def summarize(boot):
    return (np.nanmean(boot, axis=0),
            np.nanpercentile(boot, 2.5, axis=0),
            np.nanpercentile(boot, 97.5, axis=0))

bm_a, lo_a, hi_a = summarize(boot_avail)
bm_i, lo_i, hi_i = summarize(boot_imp)

rank_avail = pd.Series(-smd_avail).rank(method="min").astype(int).to_numpy()
rank_imp = pd.Series(-smd_imp).rank(method="min").astype(int).to_numpy()

robust = pd.DataFrame({
    "Feature": features3,
    "Avail_Diff": obs_diff_avail,
    "Avail_Boot_Mean": bm_a,
    "Avail_CI_Lower": lo_a,
    "Avail_CI_Upper": hi_a,
    "Avail_Abs_Std_Diff": smd_avail,
    "Avail_Rank": rank_avail,
    "Imputed_Diff": obs_diff_imp,
    "Imputed_Boot_Mean": bm_i,
    "Imputed_CI_Lower": lo_i,
    "Imputed_CI_Upper": hi_i,
    "Imputed_Abs_Std_Diff": smd_imp,
    "Imputed_Rank": rank_imp,
})
robust["Diff_Change"] = robust["Imputed_Diff"] - robust["Avail_Diff"]
robust["Abs_Diff_Change"] = robust["Diff_Change"].abs()
robust["Sign_Avail"] = np.sign(robust["Avail_Diff"]).astype(int)
robust["Sign_Imputed"] = np.sign(robust["Imputed_Diff"]).astype(int)
robust["Sign_Changed"] = robust["Sign_Avail"] != robust["Sign_Imputed"]
robust["Rank_Changed"] = robust["Avail_Rank"] != robust["Imputed_Rank"]
robust["Avail_CI_Excludes_Zero"] = (robust["Avail_CI_Lower"] > 0) | (robust["Avail_CI_Upper"] < 0)
robust["Imputed_CI_Excludes_Zero"] = (robust["Imputed_CI_Lower"] > 0) | (robust["Imputed_CI_Upper"] < 0)
robust["CI_Conclusion_Changed"] = robust["Avail_CI_Excludes_Zero"] != robust["Imputed_CI_Excludes_Zero"]

robust_path = OUTPUT_DIR / "robustness_comparison.csv"
robust.to_csv(robust_path, index=False)

top = robust.loc[robust["Abs_Diff_Change"].idxmax()]
print(robust.round(4).T.to_string(header=False))
print()
print("Selected Features:", features3)
print("Bootstrap Repetitions:", N_BOOT)
print("Largest Change in Mean Difference:", f"{top['Abs_Diff_Change']:.4f}", f"({top['Feature']})")
print("Features with Changed Difference Sign:", list(robust.loc[robust["Sign_Changed"], "Feature"]) or "None")
print("Features with Changed Ranking:", list(robust.loc[robust["Rank_Changed"], "Feature"]) or "None")
print("Reproducibility Verified:", reproducible)

Feature                   Feature_011  Feature_048  Feature_114  Feature_339  Feature_474
Avail_Diff                    -0.0016      -0.2773       0.0005       0.7294       1.4715
Avail_Boot_Mean               -0.0017      -0.3233       0.0005       0.7157       1.2371
Avail_CI_Lower                 -0.004      -1.2613      -0.0001      -0.2989      -3.4103
Avail_CI_Upper                 0.0004       0.5878       0.0012       2.4627        7.779
Avail_Abs_Std_Diff              0.131       0.0613       0.2763       0.1201       0.0593
Avail_Rank                          2            4            1            3            5
Imputed_Diff                  -0.0016      -0.2773       0.0005       0.7286        1.494
Imputed_Boot_Mean             -0.0017      -0.3233       0.0005       0.7148       1.2606
Imputed_CI_Lower               -0.004      -1.2613      -0.0001      -0.2988      -3.3859
Imputed_CI_Upper               0.0004       0.5879       0.0012       2.4612        7.797
Imputed_Ab

In [9]:
# [CELL 9] EX3 - SENSITIVITY REPORT AND OUTPUT CHECK
changed_sign = list(robust.loc[robust["Sign_Changed"], "Feature"])
changed_rank = list(robust.loc[robust["Rank_Changed"], "Feature"])
changed_ci = list(robust.loc[robust["CI_Conclusion_Changed"], "Feature"])

lines = [
    "SECOM STATISTICAL ROBUSTNESS AND SENSITIVITY REPORT",
    f"Student: {LAST_NAME}",
    f"Student ID: {STUDENT_ID}",
    f"Random Seed: {RANDOM_SEED}",
    f"Seed Digit: {SEED_DIGIT}",
    f"Bootstrap Repetitions: {N_BOOT}",
    f"Selected Features: {', '.join(features3)}",
    f"Largest Change in Mean Difference: {top['Abs_Diff_Change']:.4f} ({top['Feature']})",
    f"Features with Changed Difference Sign: {', '.join(changed_sign) if changed_sign else 'None'}",
    f"Features with Changed Ranking: {', '.join(changed_rank) if changed_rank else 'None'}",
    f"Features with Changed CI Conclusion (excludes zero): {', '.join(changed_ci) if changed_ci else 'None'}",
    f"Reproducibility Verified: {reproducible}",
    "",
    "Per-feature results (Fail - Pass mean difference, 95% percentile CI):",
]
for _, r in robust.iterrows():
    lines.append(
        f"- {r['Feature']}: available-case {r['Avail_Diff']:.4f} "
        f"[{r['Avail_CI_Lower']:.4f}, {r['Avail_CI_Upper']:.4f}] rank {int(r['Avail_Rank'])}; "
        f"median-imputed {r['Imputed_Diff']:.4f} "
        f"[{r['Imputed_CI_Lower']:.4f}, {r['Imputed_CI_Upper']:.4f}] rank {int(r['Imputed_Rank'])}"
    )
lines += [
    "",
    "Interpretation Notes:",
    "- Median imputation pulls missing values toward the center, so it can shrink group differences relative to available-case analysis.",
    "- A confidence interval that excludes zero shows a stable difference in the resampled data, not a cause of failure.",
    "- Rankings use absolute standardized differences because raw scales differ between features.",
    "- Results are descriptive and do not establish prediction accuracy.",
]
report_path = OUTPUT_DIR / "sensitivity_report.txt"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

for path in [quality_path, balanced_path, comparison_path, robust_path, report_path]:
    print(path.name, "Created:", path.is_file())

measurement_quality.csv Created: True
balanced_sample.csv Created: True
feature_comparison.csv Created: True
robustness_comparison.csv Created: True
sensitivity_report.txt Created: True
